In [1]:
import os
from glob import glob
import numpy as np
from scipy.io import wavfile
from scipy.io.wavfile import write as writewav
from vosk import Model
from vowel_functions import (
    preprocess,          
    rec_vosk,            
    extract_vowels,      
    groupedframes_to_lists,  
)

# ====== 1) Paths ======
PREPROC_DIR = r"C:\Users\Desktop\audio_preproc"     # Directory with preprocessed audio(omit the preprocessing steps here)
MODEL_PATH  = r"C:\Users\Desktop\models\vosk-model-small-sv-rhasspy-0.15"
SAVE_DIR    = r"C:\Users\Desktop\yourfile\frames_40ms"       # Output folder
os.makedirs(SAVE_DIR, exist_ok=True)

# ====== 2) Extraction parameters ======
FL_MS = 40  # Frame length in ms, can be adjusted
KW_EXTRACT = dict(
    white_thr=0.8,   # Whiteness (noise) threshold
    vol_thr=50,      # Silence threshold in dB
    zero_thr=0.5,    # Zero-crossing rate threshold
    zero_thr_2=0.05, # Secondary zero-crossing threshold
    long_frame=True, # Use 3×FL (120ms) window to detect peaks, output 40ms
    zero_pad=True,
    add_context=False,
    print_info=False,
)

# ====== 3) Utility function ======
def to_int16(x: np.ndarray) -> np.ndarray:
    """Normalize a single audio frame to int16 range."""
    x = np.asarray(x, dtype=np.float64)
    m = np.max(np.abs(x)) + 1e-12
    return (x / m * np.iinfo(np.int16).max).astype(np.int16)

# ====== 4) Load Vosk model ======
print("Loading Vosk model ...")
model = Model(MODEL_PATH)
print("Model ready.")

# ====== 5) Main loop ======
wav_list = sorted(glob(os.path.join(PREPROC_DIR, "*.wav")))
total_files = len(wav_list)
total_frames_saved = 0

for idx_file, pp_path in enumerate(wav_list, 1):
    try:
        # Read audio
        Fs, audio = wavfile.read(pp_path)
        if audio.ndim == 2:
            audio = audio[:, 0]

        # Speech recognition (no per-word printout)
        words = rec_vosk(pp_path, model, print_summary=False)

        # Extract vowel frames
        fl = int(FL_MS * 1e-3 * Fs)
        grouped_frames = extract_vowels(words, audio, Fs, fl, **KW_EXTRACT)

        # Flatten and sort by start time
        flat_items = []
        for v, D in grouped_frames.items():
            for i, fr in enumerate(D["frame"]):
                st = float(D["start"][i])
                en = float(D["stop"][i])
                flat_items.append((st, en, v, np.asarray(fr, dtype=np.float64)))
        flat_items.sort(key=lambda x: x[0])

        # Save frames
        base = os.path.splitext(os.path.basename(pp_path))[0]
        saved_this = 0
        for k, (st, en, v, fr) in enumerate(flat_items, start=1):
            out_name = f"{base}_v{v}_{k:03d}_{st:.3f}-{en:.3f}_40ms.wav"
            out_path = os.path.join(SAVE_DIR, out_name)
            fr16 = to_int16(fr)
            writewav(out_path, Fs, fr16)
            saved_this += 1

        total_frames_saved += saved_this
        print(f"[{idx_file:4d}/{total_files:4d}] {base}: saved {saved_this:3d} vowel frames")

    except Exception as e:
        print(f"[{idx_file:4d}/{total_files:4d}] {os.path.basename(pp_path)}: ERROR → {e}")

print(f"\nDone. Files processed: {total_files}, vowel frames saved: {total_frames_saved}")



Loading Vosk model ...
Model ready.
[   1/3296] A_18BTQ623_a_200309164631_pp: saved   2 vowel frames
[   2/3296] A_18BTQ623_a_200309165307_pp: saved   6 vowel frames
[   3/3296] A_1HEFA3VE_a_190808203859_pp: saved   1 vowel frames
[   4/3296] A_1HEFA3VE_a_191202201727_pp: saved   2 vowel frames
[   5/3296] A_1HEFA3VE_t_190808204034_pp: saved  50 vowel frames
[   6/3296] A_1HEFA3VE_t_191202201827_pp: saved  42 vowel frames
[   7/3296] A_1MXSNK2Q_a_191120111745_pp: saved   7 vowel frames
[   8/3296] A_1MXSNK2Q_a_200214153100_pp: saved   0 vowel frames
[   9/3296] A_1MXSNK2Q_t_191120111835_pp: saved  28 vowel frames
[  10/3296] A_1MXSNK2Q_t_200214153105_pp: saved   2 vowel frames
[  11/3296] A_1SAUDAHC_a_180928130926_pp: saved   5 vowel frames
[  12/3296] A_1SAUDAHC_a_181007141625_pp: saved   4 vowel frames
[  13/3296] A_1SAUDAHC_a_181014145336_pp: saved   2 vowel frames
[  14/3296] A_1SAUDAHC_a_181021191951_pp: saved   0 vowel frames
[  15/3296] A_1SAUDAHC_a_181029193630_pp: saved   2 vo

D:\anaconda\envs\ttf-gpu\lib\site-packages\Signal_Analysis\features\signal.py:625: RuntimeWarning: divide by zero encountered in divide
  best_cands = 10.0 * np.log10( best_cands / ( 1.0 - best_cands ) )


[  33/3296] A_36Y6DO21_t_200415230340_pp: saved  33 vowel frames
[  34/3296] A_36Y6DO21_t_200423231200_pp: saved  19 vowel frames
[  35/3296] A_36Y6DO21_t_200505181731_pp: saved  34 vowel frames
[  36/3296] A_3NI7RMPD_a_190613170242_pp: saved   1 vowel frames
[  37/3296] A_3NI7RMPD_a_190716091807_pp: saved   2 vowel frames
[  38/3296] A_3NI7RMPD_t_190716091653_pp: saved  57 vowel frames
[  39/3296] A_3NI7RMPD_t_190716092310_pp: saved  50 vowel frames
[  40/3296] A_4SACQHQV_a_200415223858_pp: saved   2 vowel frames
[  41/3296] A_4SACQHQV_a_200426232935_pp: saved   2 vowel frames
[  42/3296] A_4SACQHQV_a_200510233216_pp: saved   4 vowel frames
[  43/3296] A_4SACQHQV_t_200415224043_pp: saved  66 vowel frames
[  44/3296] A_4SACQHQV_t_200510233325_pp: saved  43 vowel frames
[  45/3296] A_4ZZN21U1_a_191002163321_pp: saved   2 vowel frames
[  46/3296] A_4ZZN21U1_a_200325123403_pp: saved   1 vowel frames
[  47/3296] A_4ZZN21U1_a_200326091900_pp: saved   1 vowel frames
[  48/3296] A_56Q4DF55_a_

C:\Users\Liam\vowel_functions.py:250: RuntimeWarning: divide by zero encountered in double_scalars
  return (sum1 / sum0 > wNoiseRatio), sum1 / sum0


[ 344/3296] A_870QORHD_a_200126164826_pp: saved   0 vowel frames
[ 345/3296] A_870QORHD_a_200203172552_pp: saved   1 vowel frames
[ 346/3296] A_870QORHD_a_200207191816_pp: saved   1 vowel frames
[ 347/3296] A_870QORHD_a_200210142013_pp: saved   0 vowel frames
[ 348/3296] A_870QORHD_a_200215132948_pp: saved   3 vowel frames
[ 349/3296] A_870QORHD_a_200222103734_pp: saved   2 vowel frames
[ 350/3296] A_870QORHD_a_200226150409_pp: saved   4 vowel frames
[ 351/3296] A_870QORHD_a_200303190857_pp: saved   7 vowel frames
[ 352/3296] A_870QORHD_a_200309091419_pp: saved   1 vowel frames
[ 353/3296] A_870QORHD_a_200312103922_pp: saved   3 vowel frames
[ 354/3296] A_870QORHD_a_200317171124_pp: saved   4 vowel frames
[ 355/3296] A_870QORHD_a_200319084503_pp: saved   2 vowel frames
[ 356/3296] A_870QORHD_a_200321084638_pp: saved   4 vowel frames
[ 357/3296] A_870QORHD_a_200322084802_pp: saved   4 vowel frames
[ 358/3296] A_870QORHD_a_200323084419_pp: saved   3 vowel frames
[ 359/3296] A_870QORHD_a_